# Singular Value Decomposition (SVD) - Starter Notebook
Explores SVD for dimensionality reduction, image compression, and latent factor discovery.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

In [ ]:
# SVD on a small matrix
A = np.array([[1, 2, 0, 4],
              [3, 1, 2, 0],
              [0, 4, 1, 3],
              [2, 0, 3, 1]], dtype=float)

U, s, Vt = np.linalg.svd(A)
S = np.diag(s)

print('Singular values:', np.round(s, 4))
print('Rank of A:', np.linalg.matrix_rank(A))
print('Reconstruction error (full SVD):', np.round(np.linalg.norm(A - U @ S @ Vt), 10))

In [ ]:
# Low-rank approximation
print('\nLow-rank approximation errors:')
for k in range(1, len(s)+1):
    A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]
    err = np.linalg.norm(A - A_k, 'fro')
    var_explained = np.sum(s[:k]**2) / np.sum(s**2)
    print(f'  k={k}: Frobenius error={err:.4f}, variance explained={var_explained:.2%}')

In [ ]:
# SVD for dimensionality reduction on digits dataset
digits = load_digits()
X = digits.data.astype(float)
X = StandardScaler().fit_transform(X)

U_d, s_d, Vt_d = np.linalg.svd(X, full_matrices=False)

# Scree plot
var_ratio = s_d**2 / np.sum(s_d**2)
cum_var = np.cumsum(var_ratio)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.bar(range(1, 21), var_ratio[:20], color='steelblue')
ax1.set_xlabel('Component')
ax1.set_ylabel('Variance explained')
ax1.set_title('Scree Plot (SVD)')

ax2.plot(range(1, len(cum_var)+1), cum_var, color='darkorange')
ax2.axhline(0.95, color='red', linestyle='--', label='95% variance')
ax2.set_xlabel('Number of singular values')
ax2.set_ylabel('Cumulative variance explained')
ax2.set_title('Cumulative Variance (SVD)')
ax2.legend()

plt.tight_layout()
plt.show()
k95 = np.searchsorted(cum_var, 0.95) + 1
print(f'Components needed for 95% variance: {k95}')

In [ ]:
# 2D projection via top-2 SVD components
Z = U_d[:, :2] * s_d[:2]

plt.figure(figsize=(7, 5))
scatter = plt.scatter(Z[:, 0], Z[:, 1], c=digits.target,
                      cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Digit class')
plt.xlabel('SVD component 1')
plt.ylabel('SVD component 2')
plt.title('Digits projected onto top-2 SVD components')
plt.tight_layout()
plt.show()